In [1]:
# Cell 1: Read silver_sales
df = spark.read.format('delta').load('Tables/dbo/silver_sales')
print(f'silver_sales rows: {df.count()}')

StatementMeta(, 65fe1bda-2371-44c1-b1e2-cba47dcf7f63, 3, Finished, Available, Finished, False)

silver_sales rows: 1268


In [2]:
# Cell 2: Build dim_customer
from pyspark.sql.functions import monotonically_increasing_id

dim_customer = df.select('customer_name', 'region') \
    .distinct() \
    .withColumn('customer_key', monotonically_increasing_id() + 1)

# Reorder columns: key first
dim_customer = dim_customer.select('customer_key', 'customer_name', 'region')

dim_customer.write.format('delta').mode('overwrite').saveAsTable('dim_customer')
print(f'dim_customer rows: {dim_customer.count()}')
display(dim_customer.limit(5))

StatementMeta(, 65fe1bda-2371-44c1-b1e2-cba47dcf7f63, 4, Finished, Available, Finished, False)

dim_customer rows: 93


SynapseWidget(Synapse.DataFrame, 9e25ecfb-9368-4dac-9249-9891053bde22)

In [3]:
# Cell 3: Build dim_product
dim_product = df.select('product_category') \
    .distinct() \
    .withColumn('product_key', monotonically_increasing_id() + 1)

# Reorder columns: key first
dim_product = dim_product.select('product_key', 'product_category')

dim_product.write.format('delta').mode('overwrite').saveAsTable('dim_product')
print(f'dim_product rows: {dim_product.count()}')
display(dim_product)

StatementMeta(, 65fe1bda-2371-44c1-b1e2-cba47dcf7f63, 5, Finished, Available, Finished, False)

dim_product rows: 3


SynapseWidget(Synapse.DataFrame, 9005d448-4093-4592-a437-d04aa81a8456)

In [4]:
# Cell 4: Build dim_region
dim_region = df.select('region') \
    .distinct() \
    .withColumn('region_key', monotonically_increasing_id() + 1)

# Reorder columns: key first
dim_region = dim_region.select('region_key', 'region')

dim_region.write.format('delta').mode('overwrite').saveAsTable('dim_region')
print(f'dim_product rows: {dim_region.count()}')
display(dim_region)

StatementMeta(, 65fe1bda-2371-44c1-b1e2-cba47dcf7f63, 6, Finished, Available, Finished, False)

dim_product rows: 4


SynapseWidget(Synapse.DataFrame, a3d7ef14-f99c-4b91-9903-f0a37270dbdb)

In [6]:
# Cell 5: Build fact_sales by joining silver_sales to each dimension 
from pyspark.sql.functions import col, to_date, date_format

# Re-read silver and all dimensions
df_silver = spark.read.format('delta').load('Tables/dbo/silver_sales')
df_customer = spark.read.format('delta').load('Tables/dbo/dim_customer')
df_product = spark.read.format('delta').load('Tables/dbo/dim_product')
df_region = spark.read.format('delta').load('Tables/dbo/dim_region')

# Join to resolve surrogate keys
fact = df_silver \
    .join (df_customer,
            (df_silver.customer_name == df_customer.customer_name) & 
            (df_silver.region == df_customer.region), 
            'left') \
    .join(df_product,
            df_silver.product_category == df_product.product_category, 
            'left') \
    .join(df_region,
            df_silver.region == df_region.region, 
            'left')

# Build the date surrogate key (YYYYMMDD integer)
fact = fact.withColumn(
    'order_date_key',
    date_format(to_date(col('order_date'), 'yyyy-MM-dd'), 'yyyyMMdd').cast('int')
)

# Select only the fact table columns
fact_sales = fact.select(
    col('order_id'),
    col('order_date_key'),
    col('customer_key'),
    col('product_key'),
    col('region_key'),
    col('revenue'),
    col('quantity'),
    col('revenue_usd')
)

fact_sales.write.format('delta').mode('overwrite').saveAsTable('fact_sales')
print(f'fact_sales rows: {fact_sales.count()}')
display(fact_sales.limit(5))

StatementMeta(, 65fe1bda-2371-44c1-b1e2-cba47dcf7f63, 8, Finished, Available, Finished, False)

fact_sales rows: 1268


SynapseWidget(Synapse.DataFrame, 79ad302f-86b7-4271-8055-3285e03cc215)